In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv()
BASE_DIR = os.getenv("BASE_DIR")

data_dir = f"{BASE_DIR}/data/batch_nov15/o"
diag_file = f"{data_dir}/diagnostics/diagnostics.csv"

df = pd.read_csv(diag_file)

In [ ]:
import os, re
import pandas as pd
import matplotlib.pyplot as plt

PREFIX = 'ECH'    # 'ECH' (herding) or 'ECK' (k-center)
AE = False        # True: plot the .AE experiments instead

directory = f"{BASE_DIR}/data/batch_nov15/o/results"
METHOD = {'ECH': 'Herding', 'ECK': 'k-center'}[PREFIX]
EMBEDDINGS = ['Position only', 'Position + piece count', 'Position + near goal',
              'Position + clear lane', 'All three']
MODELS = ['CNN', 'Res']
COLORS = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']   # one per embedding, same in every panel
METRICS = [('mse', 'MSE on all of E'), ('train_mse', 'MSE on the training subset')]

# find <PREFIX><n>.csv (or <PREFIX><n>.AE.csv) and sort by n
pattern = re.compile(rf'^{PREFIX}(\d+)' + (r'\.AE' if AE else '') + r'\.csv$')
found = sorted((int(m.group(1)), e.path) for e in os.scandir(directory)
               if e.is_file() and (m := pattern.match(e.name)))
print(f'{len(found)} files:', [os.path.basename(p) for _, p in found])

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey='col')

for n, path in found:
    emb, model = EMBEDDINGS[(n - 1) % 5], MODELS[(n - 1) // 5]
    df = pd.read_csv(path)
    for col, (metric, _) in enumerate(METRICS):
        axes[MODELS.index(model), col].plot(df['ratio'] * 100, df[metric],
                                            color=COLORS[(n - 1) % 5], lw=2, label=emb)

for row, model in enumerate(MODELS):
    for col, (_, title) in enumerate(METRICS):
        ax = axes[row, col]
        ax.set_title(f'{title} ({model})', fontsize=11)
        ax.grid(alpha=0.3, lw=0.5)
        for side in ('top', 'right'):
            ax.spines[side].set_visible(False)
        if row == 1:
            ax.set_xlabel('Training subset (% of E)')
        if col == 0:
            ax.set_ylabel('MSE')

axes[0, 0].legend(frameon=False, fontsize=9)
fig.suptitle(f'{METHOD}, o.txt: embeddings with CNN and Residual' + (' (on autoencoder codes)' if AE else ''),
             fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
import os, re
import pandas as pd
import matplotlib.pyplot as plt

PREFIX = 'ECK'    # 'ECH' (herding) or 'ECK' (k-center)
AE = False        # True: plot the .AE experiments instead

directory = f"{BASE_DIR}/data/batch_nov15/o/results"
METHOD = {'ECH': 'Herding', 'ECK': 'k-center'}[PREFIX]
EMBEDDINGS = ['Position only', 'Position + piece count', 'Position + near goal',
              'Position + clear lane', 'All three']
MODELS = ['CNN', 'Res']
COLORS = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']   # one per embedding, same in every panel
METRICS = [('mse', 'MSE on all of E'), ('train_mse', 'MSE on the training subset')]

# find <PREFIX><n>.csv (or <PREFIX><n>.AE.csv) and sort by n
pattern = re.compile(rf'^{PREFIX}(\d+)' + (r'\.AE' if AE else '') + r'\.csv$')
found = sorted((int(m.group(1)), e.path) for e in os.scandir(directory)
               if e.is_file() and (m := pattern.match(e.name)))
print(f'{len(found)} files:', [os.path.basename(p) for _, p in found])

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey='col')

for n, path in found:
    emb, model = EMBEDDINGS[(n - 1) % 5], MODELS[(n - 1) // 5]
    df = pd.read_csv(path)
    for col, (metric, _) in enumerate(METRICS):
        axes[MODELS.index(model), col].plot(df['ratio'] * 100, df[metric],
                                            color=COLORS[(n - 1) % 5], lw=2, label=emb)

for row, model in enumerate(MODELS):
    for col, (_, title) in enumerate(METRICS):
        ax = axes[row, col]
        ax.set_title(f'{title} ({model})', fontsize=11)
        ax.grid(alpha=0.3, lw=0.5)
        for side in ('top', 'right'):
            ax.spines[side].set_visible(False)
        if row == 1:
            ax.set_xlabel('Training subset (% of E)')
        if col == 0:
            ax.set_ylabel('MSE')

axes[0, 0].legend(frameon=False, fontsize=9)
fig.suptitle(f'{METHOD}, o.txt: embeddings with CNN and Residual' + (' (on autoencoder codes)' if AE else ''),
             fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
import os, re
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

AE = False          # True: the .AE experiments
ALPHA = 0.35
SHOW_MEAN = True   # True: also draw each method's mean curve, opaque

directory = f"{BASE_DIR}/data/batch_nov15/o/results"
MODELS = ['CNN', 'Res']                                   # experiments 1-5 are CNN, 6-10 are Residual
METHODS = {'ECH': ('Herding', '#2a78d6'), 'ECK': ('k-center', '#e34948')}
METRICS = [('mse', 'MSE on all of E'), ('train_mse', 'MSE on the training subset')]


def find(prefix):
    pattern = re.compile(rf'^{prefix}(\d+)' + (r'\.AE' if AE else '') + r'\.csv$')
    return sorted((int(m.group(1)), e.path) for e in os.scandir(directory)
                  if e.is_file() and (m := pattern.match(e.name)))


fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey='col')

for prefix, (name, color) in METHODS.items():
    found = find(prefix)
    print(f'{name}: {len(found)} files')
    curves = {}   # (row, col) -> list of series, for the optional mean
    for n, path in found:
        row = (n - 1) // 5                                # 0 = CNN, 1 = Res
        df = pd.read_csv(path)
        for col, (metric, _) in enumerate(METRICS):
            axes[row, col].plot(df['ratio'] * 100, df[metric], color=color, lw=1.5, alpha=ALPHA)
            curves.setdefault((row, col), []).append(df.set_index('ratio')[metric])
    if SHOW_MEAN:
        for (row, col), series in curves.items():
            mean = pd.concat(series, axis=1).mean(axis=1)
            axes[row, col].plot(mean.index * 100, mean.values, color=color, lw=2.5)

for row, model in enumerate(MODELS):
    for col, (_, title) in enumerate(METRICS):
        ax = axes[row, col]
        ax.set_title(f'{title} ({model})', fontsize=11)
        ax.grid(alpha=0.3, lw=0.5)
        for side in ('top', 'right'):
            ax.spines[side].set_visible(False)
        if row == 1:
            ax.set_xlabel('Training subset (% of E)')
        if col == 0:
            ax.set_ylabel('MSE')

axes[0, 0].legend(handles=[Line2D([], [], color=c, lw=2, label=n) for n, c in METHODS.values()],
                  frameon=False, fontsize=10)
fig.suptitle('Herding vs k-center, o.txt' + (' (on autoencoder codes)' if AE else ''), fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
import os, re
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

AE = False          # True: the .AE experiments
ALPHA = 0.35
SHOW_MEAN = True   # True: also draw the mean curve per model and method, opaque

directory = f"{BASE_DIR}/data/batch_nov15/o/results"
MODEL_COLOR = {'CNN': '#eb6834', 'Res': '#1baf7a'}        # experiments 1-5 are CNN, 6-10 are Residual
METHODS = {'ECH': ('Herding', '-'), 'ECK': ('k-center', '--')}
METRICS = [('mse', 'MSE on all of E'), ('train_mse', 'MSE on the training subset')]


def find(prefix):
    pattern = re.compile(rf'^{prefix}(\d+)' + (r'\.AE' if AE else '') + r'\.csv$')
    return sorted((int(m.group(1)), e.path) for e in os.scandir(directory)
                  if e.is_file() and (m := pattern.match(e.name)))


fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True)

for prefix, (name, style) in METHODS.items():
    found = find(prefix)
    print(f'{name}: {len(found)} files')
    curves = {}   # (model, col) -> list of series, for the optional mean
    for n, path in found:
        model = 'CNN' if n <= 5 else 'Res'
        df = pd.read_csv(path)
        for col, (metric, _) in enumerate(METRICS):
            axes[col].plot(df['ratio'] * 100, df[metric], color=MODEL_COLOR[model], ls=style,
                           lw=1.5, alpha=ALPHA)
            curves.setdefault((model, col), []).append(df.set_index('ratio')[metric])
    if SHOW_MEAN:
        for (model, col), series in curves.items():
            mean = pd.concat(series, axis=1).mean(axis=1)
            axes[col].plot(mean.index * 100, mean.values, color=MODEL_COLOR[model], ls=style, lw=2.5)

for col, (_, title) in enumerate(METRICS):
    ax = axes[col]
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Training subset (% of E)')
    ax.grid(alpha=0.3, lw=0.5)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
axes[0].set_ylabel('MSE')

handles = ([Line2D([], [], color=c, lw=2, label=m) for m, c in MODEL_COLOR.items()] +
           [Line2D([], [], color='#52514e', lw=2, ls=s, label=n) for n, s in METHODS.values()])
axes[0].legend(handles=handles, frameon=False, fontsize=10)
fig.suptitle('Herding vs k-center, CNN and Residual, o.txt' + (' (on autoencoder codes)' if AE else ''), fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ALPHA = 1.0          # only 4 curves, so no need for transparency
BASELINE = 0.7966    # constant-predictor MSE on E (1 - mean(Y)^2), printed in the job log

directory = f"{BASE_DIR}/data/board_comparison/bt6x6_fens/results"
MODEL_COLOR = {'CNN': '#eb6834', 'Residual': '#1baf7a'}
METHOD_STYLE = {'Herding': '-', 'k-center': '--'}
EXPERIMENTS = {   # id -> (method, model)
    'BC1': ('Herding',  'CNN'),
    'BC2': ('k-center', 'CNN'),
    'BC3': ('Herding',  'Residual'),
    'BC4': ('k-center', 'Residual'),
}
METRICS = [('mse', 'MSE on all of E'), ('train_mse', 'MSE on the training subset')]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True)

for name, (method, model) in EXPERIMENTS.items():
    path = f"{directory}/{name}.csv"
    if not os.path.exists(path):
        print(f'{name}: missing, skipped')
        continue
    df = pd.read_csv(path)
    for col, (metric, _) in enumerate(METRICS):
        axes[col].plot(df['ratio'] * 100, df[metric], color=MODEL_COLOR[model], ls=METHOD_STYLE[method],
                       lw=1.8, alpha=ALPHA, marker='o', ms=4)

for col, (_, title) in enumerate(METRICS):
    ax = axes[col]
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Training subset (% of E)')
    ax.grid(alpha=0.3, lw=0.5)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
axes[0].axhline(BASELINE, color='#52514e', lw=0.8, ls=':')   # predicting a constant
axes[0].text(axes[0].get_xlim()[1], BASELINE, ' constant\n predictor', va='center', fontsize=8, color='#52514e')
axes[0].set_ylabel('MSE')

handles = ([Line2D([], [], color=c, lw=2, label=m) for m, c in MODEL_COLOR.items()] +
           [Line2D([], [], color='#52514e', lw=2, ls=s, label=m) for m, s in METHOD_STYLE.items()])
axes[0].legend(handles=handles, frameon=False, fontsize=10)
fig.suptitle('Board comparison, 6x6 (bt6x6_fens), position only', fontsize=13)
plt.tight_layout()
plt.show()